## Entorno: Data, docker y spark

Se descarga la data y se configura el entorno con docker y spark

#### Installs

Instala Jupyter (`notebook`), la dependencia necesaria para poder abrir y correr este notebook

In [1]:
%pip install -q notebook


[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


#### Imports

Importa las librerías estándar (subprocess, sys, pathlib) que usan las demás celdas del notebook

In [2]:
import subprocess, sys
from pathlib import Path

#### Utilidades

Define `sh()`, la función que ejecuta comandos de shell mostrando su salida en vivo, y calcula la ruta raíz del proyecto

In [3]:
def sh(cmd):
    p = subprocess.Popen(cmd, shell=True, cwd=proyecto, stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    for linea in p.stdout:
        print(linea, end="")

    p.wait()

    if p.returncode != 0:
        raise RuntimeError(p.returncode)

proyecto = Path.cwd()
if not (proyecto / "docker-compose.yml").exists() and (proyecto.parent / "docker-compose.yml").exists():
    proyecto = proyecto.parent

### 1. Descargar data

- Ejecuta `data/download.py` para descargar los archivos JSON de contrataciones

- El script salta los meses que ya están en `data/raw` y solo baja los que faltan

In [4]:
sh(f'"{sys.executable}" data/download.py')
print(f"{len(list((proyecto / 'data' / 'raw').glob('*/*.json')))} archivos en data/raw")

36 archivos en data/raw


### 2. Verificación docker

Comprueba que el CLI de Docker y el plugin `compose` estén instalados en la máquina

In [5]:
sh("docker --version")
sh("docker compose version")

Docker version 29.7.2, build a7dcaa6
Docker Compose version v5.4.0


### 3. Iniciar cluster spark

- Levanta el cluster de Spark (master + 2 workers) y Jupyter

- Kernel picker → "Existing Jupyter Server..." → `http://localhost:8888`

In [6]:
sh("docker compose up -d")
sh("docker compose ps")

 Container dm-proj2-spark-worker-2-1 Running 
 Container dm-proj2-jupyter-1 Running 
 Container dm-proj2-spark-master-1 Running 
 Container dm-proj2-spark-worker-1 Running 
NAME                        IMAGE                                          COMMAND                  SERVICE          CREATED       STATUS                   PORTS
dm-proj2-jupyter-1          quay.io/jupyter/pyspark-notebook:spark-3.5.0   "tini -g -- start.sh…"   jupyter          4 hours ago   Up 4 hours (healthy)     0.0.0.0:4040->4040/tcp, [::]:4040->4040/tcp, 0.0.0.0:8888->8888/tcp, [::]:8888->8888/tcp
dm-proj2-spark-master-1     quay.io/jupyter/pyspark-notebook:spark-3.5.0   "tini -g -- start.sh…"   spark-master     4 hours ago   Up 4 hours (unhealthy)   0.0.0.0:8080->8080/tcp, [::]:8080->8080/tcp
dm-proj2-spark-worker-1     quay.io/jupyter/pyspark-notebook:spark-3.5.0   "tini -g -- start.sh…"   spark-worker     4 hours ago   Up 4 hours (unhealthy)   0.0.0.0:8081->8081/tcp, [::]:8081->8081/tcp
dm-proj2-spark-worke

### 4. Monitorear cluster

- Muestra el estado de los contenedores y el consumo de CPU/RAM del cluster de Spark

- UI de Spark: `http://localhost:8080` (master) y `http://localhost:4040` (job activo)

In [7]:
sh("docker compose ps")
sh("docker stats --no-stream")

NAME                        IMAGE                                          COMMAND                  SERVICE          CREATED       STATUS                   PORTS
dm-proj2-jupyter-1          quay.io/jupyter/pyspark-notebook:spark-3.5.0   "tini -g -- start.sh…"   jupyter          4 hours ago   Up 4 hours (healthy)     0.0.0.0:4040->4040/tcp, [::]:4040->4040/tcp, 0.0.0.0:8888->8888/tcp, [::]:8888->8888/tcp
dm-proj2-spark-master-1     quay.io/jupyter/pyspark-notebook:spark-3.5.0   "tini -g -- start.sh…"   spark-master     4 hours ago   Up 4 hours (unhealthy)   0.0.0.0:8080->8080/tcp, [::]:8080->8080/tcp
dm-proj2-spark-worker-1     quay.io/jupyter/pyspark-notebook:spark-3.5.0   "tini -g -- start.sh…"   spark-worker     4 hours ago   Up 4 hours (unhealthy)   0.0.0.0:8081->8081/tcp, [::]:8081->8081/tcp
dm-proj2-spark-worker-2-1   quay.io/jupyter/pyspark-notebook:spark-3.5.0   "tini -g -- start.sh…"   spark-worker-2   4 hours ago   Up 4 hours (unhealthy)   0.0.0.0:8082->8081/tcp, [::]:8082->80

### 5. Apagar cluster

Detiene y borra los contenedores, pero la data y los resultados quedan en disco

In [ ]:
# sh("docker compose down")

 Container dm-proj2-jupyter-1 Stopping 
 Container dm-proj2-spark-worker-2-1 Stopping 
 Container dm-proj2-spark-worker-2-1 Stopped 
 Container dm-proj2-spark-worker-2-1 Removing 
 Container dm-proj2-spark-worker-2-1 Removed 
 Container dm-proj2-jupyter-1 Stopped 
 Container dm-proj2-jupyter-1 Removing 
 Container dm-proj2-jupyter-1 Removed 
 Container dm-proj2-spark-worker-1 Stopping 
 Container dm-proj2-spark-worker-1 Stopped 
 Container dm-proj2-spark-worker-1 Removing 
 Container dm-proj2-spark-worker-1 Removed 
 Container dm-proj2-spark-master-1 Stopping 
 Container dm-proj2-spark-master-1 Stopped 
 Container dm-proj2-spark-master-1 Removing 
 Container dm-proj2-spark-master-1 Removed 
 Network dm-proj2_default Removing 
 Network dm-proj2_default Removed 
